<div align="center">

<img src="https://img.shields.io/badge/KaizenStat-v0.5.9-blue?style=for-the-badge" />
<img src="https://img.shields.io/badge/Python-3.8%2B-green?style=for-the-badge" />
<img src="https://img.shields.io/badge/License-MIT-lightgrey?style=for-the-badge" />

# 🚀 KaizenStat — Quickstart Guide

**From raw data to a production-ready ML model in minutes.**  
No ML expertise required. KaizenStat handles everything automatically.

</div>

---

## What does KaizenStat do?

| Step | What happens | You write |
|------|-------------|-----------|
| **Load** | Reads CSV, Excel, Parquet, JSON, Feather, or URLs | `doctor.load("data.csv")` |
| **Health** | Scores data quality 0–100, flags issues | `doctor.health()` |
| **Fix** | Auto-cleans missing values, duplicates, outliers | `doctor.fix()` |
| **Train** | Benchmarks 5+ models, picks the best | `doctor.train()` |
| **Debug** | Detects overfitting, underfitting, data leakage | `doctor.debug_model()` |
| **Explain** | Plain-English summary of what the model learned | `doctor.explain()` |
| **Improve** | Prioritised suggestions to boost accuracy | `doctor.improve()` |
| **Report** | Exports a beautiful HTML report | `doctor.report()` |

> **All 8 steps in one call:** `doctor.run()` — or use them individually for full control.

---
## ⚙️ Step 0 — Install KaizenStat

Run the cell below once. Takes ~30 seconds on Colab.

In [ ]:
# Install KaizenStat (only needed once per session)
!pip install kaizenstat==0.5.9 -q

# Reload modules so the fresh install is active
import importlib, sys
for m in [k for k in sys.modules if k.startswith("kaizenstat")]:
    del sys.modules[m]

print("✅ KaizenStat 0.5.9 ready!")

---
## 🟢 Path A — Zero Code (Recommended for beginners)

> **Best for:** Anyone who just wants a working model with no fuss.

One method call. KaizenStat loads the data, cleans it, finds the best model, explains it, and saves it — you only need to call `.predict()` at the end.

In [ ]:
from kaizenstat import DataDoctor

# ── One line: load → clean → train → explain → export ──────────────────────
model = DataDoctor.quick_train(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",
    target="Survived",   # the column you want to predict
    tune=False,          # set tune=True for better accuracy (takes longer)
)

print("\n✅ Model is ready. Use it on new data:")
print("   model.predict(new_dataframe)")

#### 🎯 Use the model on new passengers

No preprocessing needed — pass raw data directly.

In [ ]:
import pandas as pd

# Create a new passenger — just fill in what you know
new_passenger = pd.DataFrame([{
    "Pclass":  1,          # 1st class
    "Sex":     "female",   # string, no encoding needed
    "Age":     29.0,
    "SibSp":   0,
    "Parch":   0,
    "Fare":    100.0,
    "Embarked":"S",
}])

prediction  = model.predict(new_passenger)[0]
probability = model.predict_proba(new_passenger)[0]

print(f"Prediction : {'Survived ✅' if int(prediction) == 1 else 'Did not survive ❌'}")
print(f"Confidence : {max(probability):.1%}")
print(f"Survived probability: {probability[1]:.1%}")

---
## 🔵 Path B — Autopilot with Visibility

> **Best for:** Users who want to see every step but still keep the code short.

`doctor.run()` does the same full pipeline as `quick_train()` — but prints a detailed log for each step so you can follow along.

In [ ]:
from kaizenstat import DataDoctor

# Load data + register target — one line
doctor = DataDoctor(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",
    target="Survived",
)

# Run the full 8-step pipeline with a single call
doctor.run(
    tune=False,           # True = hyperparameter search (slower, more accurate)
    report=True,          # saves titanic_report.html
    export_path="titanic_model.joblib",
)

---
## 🟡 Path C — Full Control (Step by Step)

> **Best for:** Users who want to understand each stage, customise behaviour, or add their own models/checks.

Every step is a separate method call. Skip what you don't need, repeat what you do.

### 1️⃣ Load Data

Supports: `.csv`, `.tsv`, `.xlsx`, `.xls`, `.parquet`, `.json`, `.jsonl`, `.feather` — and HTTP/HTTPS URLs.

In [ ]:
from kaizenstat import DataDoctor

doctor = DataDoctor()
doctor.load("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
doctor.fit(target="Survived")   # tell it which column to predict

### 2️⃣ Data Health Check

Scores your dataset 0–100. Flags missing values, duplicates, outliers, class imbalance, and potential data leakage.

In [ ]:
health = doctor.health()
print(f"Health Score: {health.score} / 100  (Grade: {health.grade})")
# Anything below 70 = fix before training

### 3️⃣ Auto-Fix Data

`safe=True` applies only low-risk fixes (impute missing values, remove duplicates).  
Set `safe=False` to also handle outliers and high-missing columns.

In [ ]:
fixed_df = doctor.fix(safe=True)
print(f"Remaining missing values: {fixed_df.isnull().sum().sum()}")

### 4️⃣ Train the Best Model

Benchmarks multiple algorithms (Logistic Regression, Random Forest, Gradient Boosting, XGBoost, LightGBM) and returns the winner.  
Set `tune=True` to run hyperparameter search on the winner.

In [ ]:
result = doctor.train(
    cv=5,        # cross-validation folds
    tune=False,  # True = RandomizedSearchCV on the best model (~2–5 min)
)
print(f"Best model : {result.model_name}")
print(f"Test score : {result.test_score:.4f}")
print(f"Train score: {result.train_score:.4f}")
print(f"Gap        : {result.train_score - result.test_score:.4f}  (< 0.10 is healthy)")

### 5️⃣ Debug the Model

Diagnoses overfitting, underfitting, data leakage, class imbalance, and failure slices. Gives a health score and root-cause explanation.

In [ ]:
debug = doctor.debug_model()
print(f"Diagnosis : {debug.label}")
print(f"Gap       : {debug.gap:+.4f}")
print(f"Health    : {debug.health_score}/100")

### 6️⃣ Plain-English Explanation

Translates model results into human language. Great for sharing with non-technical teammates.

In [ ]:
doctor.explain()

### 7️⃣ Feature Impact

Measures how much each feature contributes to predictions. High drop = critical feature.

In [ ]:
impact = doctor.feature_impact(top_n=10)
print("\nTop features by impact:")
for feat, drop in sorted(impact.items(), key=lambda x: -x[1])[:5]:
    bar = "█" * max(1, int(drop * 200))
    print(f"  {feat:20s}  {drop:.4f}  {bar}")

### 8️⃣ Improvement Suggestions

Prioritised action list: what to fix first, and how much accuracy gain to expect.

In [ ]:
doctor.improve()

### 9️⃣ Report & Export

Save a self-contained HTML report and the trained model file (works with `joblib.load`).

In [ ]:
doctor.report(output_path="titanic_report.html")

from IPython.display import IFrame, display
display(IFrame(src="titanic_report.html", width="100%", height="550px"))

In [ ]:
# Export the model — load it anywhere, no KaizenStat required at inference time
model_path = doctor.export_model(path="titanic_model.joblib")
print(f"Model saved to: {model_path}")
print(f"Pipeline Confidence: {doctor.pipeline_confidence()} / 100")

---
## 🔧 Bonus — Add Your Own Models

You can inject any sklearn-compatible model into the benchmark.

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier

doctor.add_model("SVM",        SVC(probability=True, kernel="rbf"))
doctor.add_model("ExtraTrees", ExtraTreesClassifier(n_estimators=100, random_state=42))

# Re-train — your models compete alongside the defaults
result = doctor.train(cv=3)
print(f"Winner: {result.model_name}  ({result.test_score:.4f})")

---
## 📋 Quick Reference Card

```python
from kaizenstat import DataDoctor

# ── Zero-friction (1 line) ──────────────────────────────────────────────────
model = DataDoctor.quick_train("data.csv", target="label")
model.predict(new_df)                        # raw DataFrame, no preprocessing

# ── Autopilot (2 lines) ─────────────────────────────────────────────────────
doctor = DataDoctor("data.csv", target="label")
doctor.run()                                 # full 8-step pipeline

# ── Full control ────────────────────────────────────────────────────────────
doctor = DataDoctor()
doctor.load("data.csv")                      # CSV/Excel/Parquet/JSON/Feather/URL
doctor.fit(df, target="label")               # or pass DataFrame directly
doctor.health()                              # data quality score 0–100
doctor.fix(safe=True)                        # auto-clean
doctor.train(cv=5, tune=True)               # benchmark + best model + tuning
doctor.debug_model()                         # overfitting / leakage diagnosis
doctor.explain()                             # plain-English summary
doctor.feature_impact(top_n=10)             # which features matter most
doctor.improve()                             # prioritised suggestions
doctor.report(output_path="report.html")    # HTML report
doctor.export_model(path="model.joblib")    # save for production
doctor.pipeline_confidence()                # production-readiness score 0–100

# ── Advanced ────────────────────────────────────────────────────────────────
doctor.add_model("SVM", SVC(probability=True))   # custom model in benchmark
doctor.add_check(fn, name="my_check")             # custom validation check
doctor.train_auto(tune=True, ensemble=True)       # AutoML + stacking ensemble
doctor.detect_drift(X_train, X_test)             # distribution drift detection
doctor.dataset_difficulty()                       # how hard is this dataset?
doctor.trust_score()                              # prediction reliability score
```

---
## 🔗 Resources & Next Steps

| Resource | Link |
|----------|------|
| 🌐 **Official Website** | [kaizenstat.com](https://www.kaizenstat.com) |
| 📦 **GitHub Repository** | [github.com/kaizenstat-python/KaizenStat](https://github.com/kaizenstat-python/KaizenStat) |
| 📖 **Basic Demo** *(health + fix + train + report)* | [`notebooks/demo_basic.ipynb`](demo_basic.ipynb) |
| 📘 **Intermediate Demo** *(drift, auto-improve, trust score)* | [`notebooks/demo_intermediate.ipynb`](demo_intermediate.ipynb) |
| 🔴 **Advanced Demo** *(feature engineering, custom models, tuning, codegen)* | [`notebooks/demo_advanced.ipynb`](demo_advanced.ipynb) |
| 🐛 **Bug Reports / Feature Requests** | [GitHub Issues](https://github.com/kaizenstat-python/KaizenStat/issues) |
| 📬 **Contact** | masuddarrahaman31@gmail.com |

---

<div align="center">

Made with ❤️ by **Masuddar Rahman**  
*KaizenStat v0.5.9 · MIT License · The Data Health & ML Debugging Framework*

</div>